# Aktív tanulás

A félkarú rabló mintájára legyen adott $K$-karú játékgép, ahol, ha szerencsénk van, egy kart meghúzva valamekkora jutalomban (*reward*) részesülhetünk. Az egyes karok által adott jutalmak valószínűségi változók, amelyek ismeretlen, egymástól független eloszlásokat követnek. A célunk, hogy a karok egymás után történő húzogatásával
- Felfedezzük a problémateret, azaz megbecsüljük az egyes karok "jóságát" (*exploration*),
- Ezt a tudást kihasználva maximalizáljuk az összes jutalmunkat (*exploitation*).

Érezzük, hogy a két cél egymásnak ellentmond: a felfedezéshez szuboptimális karokat is meg kell húznunk, ezzel azonban az összes jutalmunkat csökkentjük. A karok valószínűségi természete miatt az is előfordulhat, hogy egy-egy "jó" kar éppen rossz jutalmat ad, így érdemes lehet a pillanatnyilag rossznak gondolt karokat is többször tesztelni. Ez az *exploration vs. exploitation* dilemma.

Bár az általunk vizsgált stratégiák többféle eloszlás esetében is működnek, mi a már jól ismert Bernoulli-eloszlást fogjuk használni. Azaz minden karhoz tartozik egy ismeretlen $\theta_k$ paraméter, egy húzás pedig egy Bernoulli-kísérletnek felel meg:

\begin{align}
p\left(y \mid \theta_k \right) = \mathcal{B}ern \left(y \mid \theta_k \right) = \theta_k^{y} (1-\theta_k)^{1-y},
\end{align}

ahol $y \in \left\lbrace 0,1 \right\rbrace$ a $k$. kar meghúzása után kapott jutalom. A továbbiakban jelölje $\bar y_k$ a $k.$ kar által adott jutalmak átlagát, $n_k$ azt, hogy hányszor húztuk meg a $k$. kart, $n$ pedig az összes húzások számát egészen mostanáig.

Egy stratégia fontos jellemzője a megbánás (*regret*), ami a mi esetünkben a következőképpen definiálható:

\begin{align}
\sum_{k=1}^K n_k \left(\theta^* - \theta_k \right),
\end{align}

ahol $\theta^*$ a legjobb kar jutalmának várható értéke, $\theta_k$ pedig a $k$. karé (ami most, Bernoulli-eloszlásról lévén szó, megegyezik a paraméterrel). Azaz itt valójában azt mérjük, hogy várhatóan mennyit veszítünk amiatt, hogy a stratégiánk nem mindig a legjobb kart választja.

## Stratégiák

A következő stratégiákat fogjuk implementálni:

- $\varepsilon$-greedy: adott $\varepsilon$ valószínűséggel felfedezünk, azaz véletlenszerűen válaszott kart húzunk meg; különben azt, amelyik eddig átlagosan a legnagyobb kifizetést adta (azaz $\bar y_k$ maximális).
- UCB1: először minden kart meghúzunk egyszer, majd mindig azt húzzuk, amelyik a $\bar y_k + \sqrt{\frac{2 \ln n}{n_k}}$ mennyiséget maximalizálja.
- UCB-$\alpha$: hasonlóan az előbbihez, de $\bar y_k + \sqrt{\frac{\alpha \ln n}{n_k}}$-t használunk.
- Thompson-mintavétel: minden karra felteszünk egy $\beta$ prior eloszlást, majd mintát veszünk és a maximálisat választjuk; a jutalom ismeretében a $\beta$-eloszlás frissítése a már ismert módon történik (konjugált eloszlások, tavalyi első labor).
- Bayesi UCB: hasonlóképpen fenntartunk egy $\beta$-eloszlást, de a választásnál nem mintát veszünk, hanem $1- \frac{1}{1+n}$ percentilist veszünk alapul (pl. a `beta.ppf()` függvénnyel).

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import beta

In [ ]:
# Prototípus ágens (példa)
class Agent:
  def __init__(self,K):
    self.X = [] # eddigi jutalmak
    self.A = [] # melyik kart húztuk
    self.K = K  # karok száma

  # Kötelező: melyik legyen a következő kar
  def get_arm(self):
    self.arm = np.random.choice(self.K)
    return self.arm

  # Kötelező: az ágens állapotának frissítése
  def update(self,reward):
    self.A += [self.arm]
    self.X += [reward]

**1. feladat.** Valósítsa meg az ágenseket a prototípus alapján! Az ágenseknek a `get_arm()` és az `update()` függvényeket kötelező implementálniuk.

**2. feladat.** Szimulálja mindegyik ágens választásait az alábbi karoknál!

In [ ]:
rewards = [0.01, 0.02, 0.3, 0.4, 0.5, 0.6, 0.795, 0.8, 0.805]

In [ ]:
# Szimuláció T db húzással
def simulate(rewards,agent,T=10000):
  for t in range(T):
    arm    = agent.get_arm()
    reward = (np.random.rand()<rewards[arm])*1
    agent.update(reward)

In [ ]:
agent = Agent(len(rewards))
simulate(rewards,agent)

**3. feladat.** Minden ágenshez ábrázolja scatter plot-on, mikor melyik kart húzta meg (vízszintes tengely: húzások, függőleges tengely: karok)! A plotok alapján hasonlítsa össze az ágensek felfedező tevékenységét!

Ügyeljen rá, hogy az eredmények az egyes futásoknál eltérhetnek, így célszerű többször is lefuttatni a méréseket. Különösen az $\varepsilon$-greedy stratégia mutat nagy variabilitást; magyarázza meg, hogy miért!

**4. feladat.** Értékelje ki és hasonlítsa össze az ágensek teljesítményét! A következőket vegye alapul:

- Átlagos jutalom
- A legjobb kar meghúzásának relatív frekvenciája
- Regret

Az eredményeket az idő (húzások) függvényében is ábrázolja (azaz az ágens tároljon el minden ehhez szükséges információt)! Miben különbözik az $\varepsilon$-greedy ágens regret-je a többi módszerétől?

**5. feladat.** Tetszőleges problémában ábrázolja a Thompson ágens
$\beta$-posteriorjait a tanítás legvégén! Mit lehet megfigyelni? Miért mondhatjuk, hogy ez egy "optimista" stratégia?